In [10]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [11]:
# !python -m spacy download en_core_web_md -qq
# !pip install truecase -qq
!pip install -qq gensim
!pip install pyLDAvis -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 24.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [48]:
import numpy as np
import pandas as pd
import spacy
import en_core_web_md
# import truecase
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
import pyLDAvis.gensim_models
pyLDAvis.enable_notebook()
import gensim
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from sklearn.model_selection import train_test_split

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
headlines = pd.read_csv('/content/abcnews-date-text.csv')
headlines

,publish_date,headline_text
0,20030219,aba decides against community broadcasting licence
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers
...,...,...
1244179,20211231,two aged care residents die as state records 2;093
1244180,20211231,victoria records 5;919 new cases and seven deaths
1244181,20211231,wa delays adopting new close contact definition
1244182,20211231,western ringtail possums found badly dehydrated in heatwave


In [18]:
pd.set_option('display.max_colwidth', None)

In [19]:
# sample = headlines.sample(n=10, random_state=111)
# sample

In [21]:
# nltk.download('punkt_tab')

In [22]:
# truecase

# sample['truecase_headlines'] = sample['headline_text'].apply(truecase.get_true_case)

In [24]:
# sample

In [25]:
# test_truecase = np.array(sample.get('truecase_headlines'))
# for doc in nlp.pipe(test_truecase):
#     print([(ent.text, ent.label_) for ent in doc.ents])

In [26]:
# test_headlines = np.array(sample.get('headline_text'))
# for doc in nlp.pipe(test_headlines):
#     print([(ent.text, ent.label_) for ent in doc.ents])

In [31]:
X_train, X_test = train_test_split(headlines, train_size=50000, random_state=111)

In [32]:
X_train = X_train.reset_index()

In [33]:
nlp = spacy.load('en_core_web_md', disable=['parser', 'ner'])

In [40]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
  tokens = []
  for headline in nlp.pipe(texts):
    new_text = [token.lemma_ for token in headline if token.pos_ in allowed_postags and not token.is_stop and token.is_alpha]
    tokens.append(new_text)

  return tokens

lemmatized_texts = lemmatization(X_train['headline_text'])

In [41]:
lemmatized_texts[1][0:90]

['elder', 'sale', 'lead', 'rural', 'branch', 'closure']

In [52]:
bigrams_phrases = gensim.models.Phrases(lemmatized_texts, min_count=5, threshold=50)
trigrams_phrases = gensim.models.Phrases(bigrams_phrases[lemmatized_texts], threshold=50)

bigram = gensim.models.phrases.Phraser(bigrams_phrases)
trigram = gensim.models.phrases.Phraser(trigrams_phrases)

def make_bigrams(texts):
  return [bigram[doc] for doc in texts]

def make_trigrams(texts):
  return [trigram[bigram[doc]] for doc in texts]

data_bigrams = make_bigrams(lemmatized_texts)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print(data_bigrams_trigrams)

[['new', 'high', 'school', 'open', 'early'], ['elder', 'sale', 'lead', 'rural', 'branch', 'closure'], ['oppose', 'ban', 'bump', 'stock', 'gun', 'device'], ['south', 'australian', 'talent', 'grab', 'draft'], ['trump', 'tweet', 'nice', 'note', 'troop', 'remain'], ['ethical', 'investment', 'small', 'grow'], ['nrl_live_streaming', 'update'], ['vodka', 'win', 'international', 'award'], ['indefinite', 'jail', 'answer', 'women', 'group'], ['face', 'terrorism', 'fund', 'risk', 'profit', 'austrac'], ['reuter', 'journalist', 'plead_guilty', 'court'], ['win', 'clipsall', 'race'], ['man', 'jail', 'newsagency', 'hold'], ['hit', 'kid', 'theme', 'park', 'treat'], ['french', 'flex', 'diplomatic', 'muscle', 'meeting'], ['redc', 'lament', 'thiess', 'closure'], ['economic', 'summit', 'build', 'business_confidence'], ['australian', 'open', 'action'], ['union', 'seek', 'wine', 'job_loss', 'answer'], ['showground', 'house', 'm', 'exhibition', 'centre'], ['correspondent', 'fall', 'toxic', 'sludge', 'dump'], 

In [43]:
id2word = Dictionary(lemmatized_texts)
corpus = [id2word.doc2bow(text) for text in lemmatized_texts]

corpus[0][0:90]

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]

In [46]:
lda_model = LdaModel(corpus=corpus,
                     id2word=id2word,
                     iterations=50,
                     num_topics=10,
                     random_state=111,
                     update_every=1,
                     chunksize=100,
                     passes=10)

In [47]:
lda_display = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word)
pyLDAvis.display(lda_display)